In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_path = '/content/drive/MyDrive/itda/yelpzip.csv'
df =  pd.read_csv(file_path)

In [4]:
df.head()

,Unnamed: 0,user_id,prod_id,rating,label,date,text,tag
0,0,5044,0,1.0,-1,2014-11-16,"Drinks were bad, the hot chocolate was watered...",fake
1,1,5045,0,1.0,-1,2014-09-08,This was the worst experience I've ever had a ...,fake
2,2,5046,0,3.0,-1,2013-10-06,This is located on the site of the old Spruce ...,fake
3,3,5047,0,5.0,-1,2014-11-30,I enjoyed coffee and breakfast twice at Toast ...,fake
4,4,5048,0,5.0,-1,2014-08-28,I love Toast! The food choices are fantastic -...,fake


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608458 entries, 0 to 608457
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Unnamed: 0  608458 non-null  int64  
 1   user_id     608458 non-null  int64  
 2   prod_id     608458 non-null  int64  
 3   rating      608458 non-null  float64
 4   label       608458 non-null  int64  
 5   date        608458 non-null  object 
 6   text        608458 non-null  object 
 7   tag         608458 non-null  object 
dtypes: float64(1), int64(4), object(3)
memory usage: 37.1+ MB


전처리

In [7]:
# 1. 데이터 복사
df_clean = df.copy()

# 2. unnamed 칼럼 제거
if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Unnamed: 0'])

# 4. label 변환
# 사기 -1, 정상 1 에서
# 사기 1, 정상 0으로 변환

if set(df_clean['label'].unique()) == set([-1, 1]):
    df_clean['label'] = df_clean['label'].replace({-1: 1, 1: 0})

print("\n변환 후 label 값:")
print(df_clean['label'].value_counts())

# 5. date를 datetime으로 변환
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')


# date 변환 실패 행 제거
df_clean = df_clean.dropna(subset=['date'])

# 6. 결측치 확인
print("\n결측치 개수:")
print(df_clean.isna().sum())

# 결측치 있으면 제거
essential_cols = ['user_id', 'prod_id', 'rating', 'label', 'date', 'text']

df_clean = df_clean.dropna(subset=essential_cols)

# 7. tag 제거
if 'tag' in df_clean.columns:
    df_clean = df_clean.drop(columns=['tag'])

# 8. text를 문자열로 통일
df_clean['text'] = df_clean['text'].astype(str)

# 9. id 컬럼 정리
df_clean['user_id'] = df_clean['user_id'].astype(int)
df_clean['prod_id'] = df_clean['prod_id'].astype(int)
df_clean['label'] = df_clean['label'].astype(int)

# 10. review_id 생성
# 이후 node 번호로 쓰기 위해 필요
df_clean = df_clean.reset_index(drop=True)
df_clean['review_id'] = df_clean.index

# 11. 최종 확인
print("전처리 후 데이터 크기:", df_clean.shape)
print(df_clean.head())

print("\n최종 label 분포:")
print(df_clean['label'].value_counts())

print("\n최종 컬럼:")
print(df_clean.columns)


변환 후 label 값:
label
0    528019
1     80439
Name: count, dtype: int64

결측치 개수:
user_id    0
prod_id    0
rating     0
label      0
date       0
text       0
tag        0
dtype: int64
전처리 후 데이터 크기: (608458, 7)
   user_id  prod_id  rating  label       date  \
0     5044        0     1.0      1 2014-11-16   
1     5045        0     1.0      1 2014-09-08   
2     5046        0     3.0      1 2013-10-06   
3     5047        0     5.0      1 2014-11-30   
4     5048        0     5.0      1 2014-08-28   

                                                text  review_id  
0  Drinks were bad, the hot chocolate was watered...          0  
1  This was the worst experience I've ever had a ...          1  
2  This is located on the site of the old Spruce ...          2  
3  I enjoyed coffee and breakfast twice at Toast ...          3  
4  I love Toast! The food choices are fantastic -...          4  

최종 label 분포:
label
0    528019
1     80439
Name: count, dtype: int64

최종 컬럼:
Index(['user_id', 'pr

In [8]:
df_clean.head()

,user_id,prod_id,rating,label,date,text,review_id
0,5044,0,1.0,1,2014-11-16,"Drinks were bad, the hot chocolate was watered...",0
1,5045,0,1.0,1,2014-09-08,This was the worst experience I've ever had a ...,1
2,5046,0,3.0,1,2013-10-06,This is located on the site of the old Spruce ...,2
3,5047,0,5.0,1,2014-11-30,I enjoyed coffee and breakfast twice at Toast ...,3
4,5048,0,5.0,1,2014-08-28,I love Toast! The food choices are fantastic -...,4


In [9]:
#  전처리 파일 저장
save_path = '/content/drive/MyDrive/itda/yelpzip_preprocessed.csv'

df_clean.to_csv(save_path, index=False, encoding='utf-8-sig')

print(save_path)

/content/drive/MyDrive/itda/yelpzip_preprocessed.csv
